<a href="https://colab.research.google.com/github/anisrabiu/PINN-2D-model/blob/main/33_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PyTorch PINN for 3-D incompressible Navier-Stokes in a single-span gothic-roof greenhouse.
# Geometry: 18 m (length, z) x 6.5 m (width, x) x 4.5 m (height, y).
#Roof: circular arc (0,1.9)-(6.5,1.9) to (3.25,4.0) then V-ridge to apex (3.25,4.5).
#Side vents: x=0 and x=6.5, y in [0.3, 1.2], z in [0, 18]. Roof vent at apex, width 0.9 m.
#Vents open 08:00–18:00; no-slip otherwise.

#05/03/2026
#Modifying the greenhouse geometry
# can you improve the greenhouse geometry, the roof, walls are transparent and the edge line with a black thin line
# to show the geometry structure of the greenhouse. Also the side vent show show how the input air.
#

In [ ]:
import torch
import math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os

# ---------- Geometry constants (meters) ----------
L   = 18.0                    # Length (z-direction)
W   = 6.5                     # Breadth (x-direction)
H   = 4.5                     # Height (y-direction)
x_c = W / 2.0                 # 3.25 m
y_e = 1.9                     # eave height at x=0 and x=W
y_mid_arc = 4.0               # arc value at x = x_c
y_apex    = H                 # 4.5 m

# Circular arc through (0,1.9), (x_c,4.0), (W,1.9); center (x_c, y_c), R ≈ 3.565 m
y_c = 0.435
R   = y_mid_arc - y_c         # ~3.565 m

# Half-width of V-ridge (m)
s_v = 0.56

# Vent geometry
SIDE_VENT_Y_LO, SIDE_VENT_Y_HI = 0.3, 1.2   # y in [0.3, 1.2] at x=0 and x=W
ROOF_VENT_HALF_WIDTH = 0.45                  # 0.9 m total at apex

# Reference scales (nondimensionalization)
L_ref = W
U_ref = 1.0                   # m/s
T_ref = L_ref / U_ref         # time scale
# Nondim: x_nd = x/L_ref, t_nd = t/T_ref, u_nd = u/U_ref, p_nd = p/(rho*U_ref^2)
# One day in seconds
DAY_SEC = 86400
# Vent open: 08:00–18:00 -> 28800 s to 64800 s
T_VENT_OPEN, T_VENT_CLOSE = 8 * 3600, 18 * 3600

# Kinematic viscosity of air (m^2/s), Re = U_ref*L_ref/nu
NU = 1.5e-5
Re = U_ref * L_ref / NU

def to_nd(x_m, y_m, z_m, t_sec=None):
    if t_sec is None:
        return x_m/L_ref, y_m/L_ref, z_m/L_ref
    return x_m/L_ref, y_m/L_ref, z_m/L_ref, t_sec/T_ref

def to_m(x_nd, y_nd, z_nd, t_nd=None):
    if t_nd is None:
        return x_nd*L_ref, y_nd*L_ref, z_nd*L_ref
    return x_nd*L_ref, y_nd*L_ref, z_nd*L_ref, t_nd*T_ref

# ---- Roof geometry (y = height as function of x) ----
def y_arc_m(x_m: torch.Tensor) -> torch.Tensor:
    return y_c + torch.sqrt(torch.clamp(R**2 - (x_m - x_c)**2, min=0.0))

def dy_arc_dx_m(x_m: torch.Tensor) -> torch.Tensor:
    denom = torch.sqrt(torch.clamp(R**2 - (x_m - x_c)**2, min=1e-12))
    return -(x_m - x_c) / denom

def y_V_left_m(x_m: torch.Tensor) -> torch.Tensor:
    x0 = x_c - s_v
    y0 = y_arc_m(torch.tensor([x0], dtype=x_m.dtype, device=x_m.device))[0]
    m  = (y_apex - y0) / (x_c - x0)
    return y0 + m * (x_m - x0)

def y_V_right_m(x_m: torch.Tensor) -> torch.Tensor:
    x1 = x_c + s_v
    y1 = y_arc_m(torch.tensor([x1], dtype=x_m.dtype, device=x_m.device))[0]
    m  = (y_apex - y1) / (x_c - x1)
    return y1 + m * (x_m - x1)

def dy_V_left_dx_m(x_m):
    x0 = x_c - s_v
    y0 = y_arc_m(torch.tensor([x0], dtype=x_m.dtype, device=x_m.device))[0]
    return torch.full_like(x_m, (y_apex - y0)/(x_c - x0))

def dy_V_right_dx_m(x_m):
    x1 = x_c + s_v
    y1 = y_arc_m(torch.tensor([x1], dtype=x_m.dtype, device=x_m.device))[0]
    return torch.full_like(x_m, (y_apex - y1)/(x_c - x1))

def y_roof_m(x_m: torch.Tensor) -> torch.Tensor:
    left  = x_m <= (x_c - s_v)
    right = x_m >= (x_c + s_v)
    midL  = (x_m > (x_c - s_v)) & (x_m <= x_c)
    midR  = (x_m > x_c) & (x_m < (x_c + s_v))
    y = torch.empty_like(x_m)
    y[left]  = y_arc_m(x_m[left])
    y[right] = y_arc_m(x_m[right])
    y[midL]  = y_V_left_m(x_m[midL])
    y[midR]  = y_V_right_m(x_m[midR])
    return y

def dy_roof_dx_m(x_m: torch.Tensor) -> torch.Tensor:
    left  = x_m <= (x_c - s_v)
    right = x_m >= (x_c + s_v)
    midL  = (x_m > (x_c - s_v)) & (x_m <= x_c)
    midR  = (x_m > x_c) & (x_m < (x_c + s_v))
    dy = torch.empty_like(x_m)
    dy[left]  = dy_arc_dx_m(x_m[left])
    dy[right] = dy_arc_dx_m(x_m[right])
    dy[midL]  = dy_V_left_dx_m(x_m[midL])
    dy[midR]  = dy_V_right_dx_m(x_m[midR])
    return dy

def roof_normal_tangent(x_m: torch.Tensor):
    y_m = y_roof_m(x_m)
    dy_dx = dy_roof_dx_m(x_m)
    tnorm = torch.sqrt(1.0 + dy_dx**2)
    tx, ty = 1.0/tnorm, dy_dx/tnorm
    nx, ny = -dy_dx/tnorm, 1.0/tnorm
    return y_m, (nx, ny), (tx, ty)

def vents_open(t_sec: torch.Tensor) -> torch.Tensor:
    """True where t is in [T_VENT_OPEN, T_VENT_CLOSE] (one day cycle)."""
    t_day = t_sec % DAY_SEC
    return (t_day >= T_VENT_OPEN) & (t_day <= T_VENT_CLOSE)

# ---- Sampling ----
def sample_interior(N, device):
    x_m = torch.rand(N, 1, device=device) * W
    z_m = torch.rand(N, 1, device=device) * L
    y_roof = y_roof_m(x_m)
    y_m = torch.rand(N, 1, device=device) * (y_roof - 1e-4) + 1e-4  # avoid y=0
    t_sec = torch.rand(N, 1, device=device) * DAY_SEC
    x_nd = x_m / L_ref
    y_nd = y_m / L_ref
    z_nd = z_m / L_ref
    t_nd = t_sec / T_ref
    return torch.cat([x_nd, y_nd, z_nd, t_nd], dim=1).requires_grad_(True)

def sample_floor(N, device):
    x_m = torch.rand(N, 1, device=device) * W
    z_m = torch.rand(N, 1, device=device) * L
    y_m = torch.zeros(N, 1, device=device)
    t_sec = torch.rand(N, 1, device=device) * DAY_SEC
    X = torch.cat([x_m/L_ref, y_m/L_ref, z_m/L_ref, t_sec/T_ref], dim=1).requires_grad_(True)
    return X

def sample_wall_x0(N, device):
    """Wall at x=0: side vent region y in [0.3,1.2], z in [0,18]; rest no-slip."""
    x_m = torch.zeros(N, 1, device=device)
    y_m = torch.rand(N, 1, device=device) * H
    z_m = torch.rand(N, 1, device=device) * L
    t_sec = torch.rand(N, 1, device=device) * DAY_SEC
    X = torch.cat([x_m/L_ref, y_m/L_ref, z_m/L_ref, t_sec/T_ref], dim=1).requires_grad_(True)
    in_vent = (y_m >= SIDE_VENT_Y_LO) & (y_m <= SIDE_VENT_Y_HI)
    return X, in_vent

def sample_wall_xW(N, device):
    x_m = torch.full((N, 1), W, device=device)
    y_m = torch.rand(N, 1, device=device) * H
    z_m = torch.rand(N, 1, device=device) * L
    t_sec = torch.rand(N, 1, device=device) * DAY_SEC
    X = torch.cat([x_m/L_ref, y_m/L_ref, z_m/L_ref, t_sec/T_ref], dim=1).requires_grad_(True)
    in_vent = (y_m >= SIDE_VENT_Y_LO) & (y_m <= SIDE_VENT_Y_HI)
    return X, in_vent

def sample_roof(N, device):
    x_m = torch.rand(N, 1, device=device) * W
    z_m = torch.rand(N, 1, device=device) * L
    y_m, (nx, ny), _ = roof_normal_tangent(x_m)
    t_sec = torch.rand(N, 1, device=device) * DAY_SEC
    X = torch.cat([x_m/L_ref, y_m/L_ref, z_m/L_ref, t_sec/T_ref], dim=1).requires_grad_(True)
    in_roof_vent = (x_m >= (x_c - ROOF_VENT_HALF_WIDTH)) & (x_m <= (x_c + ROOF_VENT_HALF_WIDTH))
    geom = {'nx': nx, 'ny': ny, 'nz': torch.zeros_like(nx)}
    return X, in_roof_vent, geom

def sample_wall_z0(N, device):
    x_m = torch.rand(N, 1, device=device) * W
    y_m = torch.rand(N, 1, device=device) * H
    z_m = torch.zeros(N, 1, device=device)
    t_sec = torch.rand(N, 1, device=device) * DAY_SEC
    return torch.cat([x_m/L_ref, y_m/L_ref, z_m/L_ref, t_sec/T_ref], dim=1).requires_grad_(True)

def sample_wall_zL(N, device):
    x_m = torch.rand(N, 1, device=device) * W
    y_m = torch.rand(N, 1, device=device) * H
    z_m = torch.full((N, 1), L, device=device)
    t_sec = torch.rand(N, 1, device=device) * DAY_SEC
    return torch.cat([x_m/L_ref, y_m/L_ref, z_m/L_ref, t_sec/T_ref], dim=1).requires_grad_(True)

def sample_initial(N, device):
    x_m = torch.rand(N, 1, device=device) * W
    z_m = torch.rand(N, 1, device=device) * L
    y_roof = y_roof_m(x_m)
    y_m = torch.rand(N, 1, device=device) * (y_roof - 1e-4) + 1e-4
    t_nd = torch.zeros(N, 1, device=device)
    return torch.cat([x_m/L_ref, y_m/L_ref, z_m/L_ref, t_nd], dim=1).requires_grad_(True)

# ---- MLP ----
class MLP(torch.nn.Module):
    def __init__(self, layers=None):
        super().__init__()
        if layers is None:
            layers = [4, 64, 64, 64, 64, 64, 64, 64, 64, 4]  # 8 hidden x 64
        self.layers = torch.nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(torch.nn.Linear(layers[i], layers[i+1]))
        self.act = torch.nn.Tanh()

    def forward(self, x):
        for i, layer in enumerate(self.layers[:-1]):
            x = self.act(layer(x))
        x = self.layers[-1](x)
        return x

# ---- Physics: 3D incompressible NS (nondimensional) ----
# u_t + u u_x + v u_y + w u_z = -p_x + (1/Re)(u_xx+u_yy+u_zz)
# v_t + u v_x + v v_y + w v_z = -p_y + (1/Re)(v_xx+...)
# w_t + u w_x + v w_y + w w_z = -p_z + (1/Re)(w_xx+...)
# u_x + v_y + w_z = 0

def compute_ns_residual(net, X, Re_nondim):
    x_nd, y_nd, z_nd, t_nd = X[:, 0:1], X[:, 1:2], X[:, 2:3], X[:, 3:4]
    out = net(X)
    u, v, w, p = out[:, 0:1], out[:, 1:2], out[:, 2:3], out[:, 3:4]
    if X.requires_grad:
        u_x  = torch.autograd.grad(u.sum(), X, create_graph=True)[0][:, 0:1]
        u_y  = torch.autograd.grad(u.sum(), X, create_graph=True)[0][:, 1:2]
        u_z  = torch.autograd.grad(u.sum(), X, create_graph=True)[0][:, 2:3]
        u_t  = torch.autograd.grad(u.sum(), X, create_graph=True)[0][:, 3:4]
        v_x  = torch.autograd.grad(v.sum(), X, create_graph=True)[0][:, 0:1]
        v_y  = torch.autograd.grad(v.sum(), X, create_graph=True)[0][:, 1:2]
        v_z  = torch.autograd.grad(v.sum(), X, create_graph=True)[0][:, 2:3]
        v_t  = torch.autograd.grad(v.sum(), X, create_graph=True)[0][:, 3:4]
        w_x  = torch.autograd.grad(w.sum(), X, create_graph=True)[0][:, 0:1]
        w_y  = torch.autograd.grad(w.sum(), X, create_graph=True)[0][:, 1:2]
        w_z  = torch.autograd.grad(w.sum(), X, create_graph=True)[0][:, 2:3]
        w_t  = torch.autograd.grad(w.sum(), X, create_graph=True)[0][:, 3:4]
        p_x  = torch.autograd.grad(p.sum(), X, create_graph=True)[0][:, 0:1]
        p_y  = torch.autograd.grad(p.sum(), X, create_graph=True)[0][:, 1:2]
        p_z  = torch.autograd.grad(p.sum(), X, create_graph=True)[0][:, 2:3]
    else:
        return None, None, None, None
    # Laplacians
    u_xx = torch.autograd.grad(u_x.sum(), X, create_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y.sum(), X, create_graph=True)[0][:, 1:2]
    u_zz = torch.autograd.grad(u_z.sum(), X, create_graph=True)[0][:, 2:3]
    v_xx = torch.autograd.grad(v_x.sum(), X, create_graph=True)[0][:, 0:1]
    v_yy = torch.autograd.grad(v_y.sum(), X, create_graph=True)[0][:, 1:2]
    v_zz = torch.autograd.grad(v_z.sum(), X, create_graph=True)[0][:, 2:3]
    w_xx = torch.autograd.grad(w_x.sum(), X, create_graph=True)[0][:, 0:1]
    w_yy = torch.autograd.grad(w_y.sum(), X, create_graph=True)[0][:, 1:2]
    w_zz = torch.autograd.grad(w_z.sum(), X, create_graph=True)[0][:, 2:3]
    lap_u = u_xx + u_yy + u_zz
    lap_v = v_xx + v_yy + v_zz
    lap_w = w_xx + w_yy + w_zz
    # Residuals
    f_u = u_t + u*u_x + v*u_y + w*u_z + p_x - (1.0/Re_nondim)*lap_u
    f_v = v_t + u*v_x + v*v_y + w*v_z + p_y - (1.0/Re_nondim)*lap_v
    f_w = w_t + u*w_x + v*w_y + w*w_z + p_z - (1.0/Re_nondim)*lap_w
    div_u = u_x + v_y + w_z
    return f_u, f_v, f_w, div_u

# ---- Data ----
def load_velocity_data(csv_path):
    data = np.loadtxt(csv_path, delimiter=',', skiprows=1)
    # Columns: x, y, z, t, S1  -> file has (x, z, y) as (3.25, 10, 1.2) so x, length, height
    x_m = torch.tensor(data[:, 0], dtype=torch.float32).reshape(-1, 1)
    z_m = torch.tensor(data[:, 1], dtype=torch.float32).reshape(-1, 1)  # length
    y_m = torch.tensor(data[:, 2], dtype=torch.float32).reshape(-1, 1)  # height
    t_sec = torch.tensor(data[:, 3], dtype=torch.float32).reshape(-1, 1)
    vel  = torch.tensor(data[:, 4], dtype=torch.float32).reshape(-1, 1)  # S1 (e.g. magnitude or u)
    x_nd = x_m / L_ref
    y_nd = y_m / L_ref
    z_nd = z_m / L_ref
    t_nd = t_sec / T_ref
    X = torch.cat([x_nd, y_nd, z_nd, t_nd], dim=1)
    return X, vel

# ---- Training ----
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # Use os.getcwd() instead of os.path.dirname(__file__) for Colab compatibility
    base_dir = os.getcwd()
    out_dir = os.path.join(base_dir, 'outputs_greenhouse_PINN')
    os.makedirs(out_dir, exist_ok=True)

    # Data (probe at x=3.25, z=10, y=1.2)
    csv_path = os.path.join(base_dir, 'wind velocity 20230228.csv')
    if os.path.isfile(csv_path):
        X_data, vel_data = load_velocity_data(csv_path)
        X_data = X_data.to(device)
        vel_data = vel_data.to(device)
        # Scale velocity to nondim (optional: match U_ref)
        vel_nd = vel_data / U_ref
    else:
        X_data = vel_nd = None

    net = MLP().to(device)
    optim = torch.optim.Adam(net.parameters(), lr=1e-3)
    Re_nd = Re  # use same Re in nondim form

    N_int, N_bc = 4000, 1500
    N_epochs = 100
    lambda_phys = 1.0
    lambda_bc   = 1.0
    lambda_ic   = 1.0
    lambda_data = 1.0

    # Probe point (width 3.25 m, length 10 m, height 1.2 m) -> (x_nd, y_nd, z_nd)
    probe_x_nd = 3.25 / L_ref
    probe_y_nd = 1.2 / L_ref
    probe_z_nd = 10.0 / L_ref

    # Time grid for probe plot (one day, every 10 min = 600 s)
    t_plot_sec = np.linspace(0, DAY_SEC, 145)  # 0 to 86400, 145 points
    t_plot_nd = torch.tensor(t_plot_sec / T_ref, dtype=torch.float32, device=device).reshape(-1, 1)
    X_probe = torch.cat([
        torch.full_like(t_plot_nd, probe_x_nd),
        torch.full_like(t_plot_nd, probe_y_nd),
        torch.full_like(t_plot_nd, probe_z_nd),
        t_plot_nd
    ], dim=1)

    # Times for velocity field plots: 06:00, 12:00, 16:00, 20:00
    times_plot_sec = [6*3600, 12*3600, 16*3600, 20*3600]
    times_plot_labels = ['06:00', '12:00', '16:00', '20:00']

    for epoch in range(1, N_epochs + 1):
        net.train()
        optim.zero_grad()

        # Interior collocation
        X_i = sample_interior(N_int, device)
        f_u, f_v, f_w, div_u = compute_ns_residual(net, X_i, Re_nd)
        loss_phys = (f_u.pow(2).mean() + f_v.pow(2).mean() + f_w.pow(2).mean() + div_u.pow(2).mean())

        # Floor: no-slip
        X_floor = sample_floor(N_bc, device)
        out_floor = net(X_floor)
        loss_floor = (out_floor[:, 0].pow(2).mean() + out_floor[:, 1].pow(2).mean() + out_floor[:, 2].pow(2).mean())

        # Wall x=0: no-slip or inflow in vent when open
        X_0, in_vent_0 = sample_wall_x0(N_bc, device)
        out_0 = net(X_0)
        t_sec_0 = X_0[:, 3:4] * T_ref
        open_0 = vents_open(t_sec_0).float()
        # No-slip where closed or outside vent; where open and in vent: u = U_in (nondim)
        U_in_nd = 0.3
        u_target_0 = open_0 * (X_0[:, 1:2] >= SIDE_VENT_Y_LO/L_ref) * (X_0[:, 1:2] <= SIDE_VENT_Y_HI/L_ref) * U_in_nd
        loss_x0 = ((out_0[:, 0:1] - u_target_0).pow(2).mean() +
                   out_0[:, 1].pow(2).mean() + out_0[:, 2].pow(2).mean())

        # Wall x=W: no-slip or outflow in vent
        X_W, _ = sample_wall_xW(N_bc, device)
        out_W = net(X_W)
        t_sec_W = X_W[:, 3:4] * T_ref
        open_W = vents_open(t_sec_W).float()
        u_target_W = -open_W * (X_W[:, 1:2] >= SIDE_VENT_Y_LO/L_ref) * (X_W[:, 1:2] <= SIDE_VENT_Y_HI/L_ref) * U_in_nd
        loss_xW = ((out_W[:, 0:1] - u_target_W).pow(2).mean() +
                   out_W[:, 1].pow(2).mean() + out_W[:, 2].pow(2).mean())

        # Roof: no-slip when closed; when open, roof vent has outflow u_n = U_out, rest no-slip
        X_roof, in_roof_vent, geom = sample_roof(N_bc, device)
        out_roof = net(X_roof)
        t_roof = X_roof[:, 3:4] * T_ref
        open_roof = vents_open(t_roof).float()
        in_roof_vent_f = in_roof_vent.float()
        nx, ny, nz = geom['nx'], geom['ny'], geom['nz']
        u_n = out_roof[:, 0:1]*nx + out_roof[:, 1:2]*ny + out_roof[:, 2:3]*nz
        U_out_nd = 0.2
        u_n_target = open_roof * in_roof_vent_f * U_out_nd
        vel_sq = out_roof[:, 0].pow(2) + out_roof[:, 1].pow(2) + out_roof[:, 2].pow(2)
        open_r = open_roof.squeeze(1)
        in_vent = in_roof_vent_f.squeeze(1)
        loss_roof = ((1 - open_r) * vel_sq).mean()
        loss_roof = loss_roof + (open_r * in_vent * (u_n.squeeze(1) - U_out_nd).pow(2)).mean()
        loss_roof = loss_roof + (open_r * (1 - in_vent) * vel_sq).mean()

        # Walls z=0, z=L: no-slip
        X_z0 = sample_wall_z0(N_bc, device)
        X_zL = sample_wall_zL(N_bc, device)
        loss_z0 = (net(X_z0)[:, :3].pow(2).mean())
        loss_zL = (net(X_zL)[:, :3].pow(2).mean())

        # Initial condition: quiescent
        X_ic = sample_initial(N_bc, device)
        out_ic = net(X_ic)
        loss_ic = (out_ic[:, 0].pow(2).mean() + out_ic[:, 1].pow(2).mean() + out_ic[:, 2].pow(2).mean())

        loss_bc = loss_floor + loss_x0 + loss_xW + loss_roof + loss_z0 + loss_zL
        loss = lambda_phys * loss_phys + lambda_bc * loss_bc + lambda_ic * loss_ic

        if X_data is not None and vel_nd is not None:
            pred = net(X_data)[:, 0:1]  # match u or magnitude
            loss_data = ((pred - vel_nd).pow(2).mean())
            loss = loss + lambda_data * loss_data
        else:
            loss_data = torch.tensor(0.0, device=device)

        loss.backward()
        optim.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{N_epochs}  loss_phys={loss_phys.item():.4e}  loss_bc={loss_bc.item():.4e}  loss_ic={loss_ic.item():.4e}  loss_data={loss_data.item():.4e}  loss={loss.item():.4e}")

            # Save state
            torch.save({'epoch': epoch, 'net': net.state_dict(), 'optim': optim.state_dict()},
                       os.path.join(out_dir, f'checkpoint_epoch_{epoch}.pt'))

            # Probe u(t) - Keep as is (time-series plot)
            with torch.no_grad():
                u_probe = net(X_probe)[:, 0].cpu().numpy()
            fig, ax = plt.subplots(1, 1, figsize=(10, 4))
            ax.plot(t_plot_sec / 3600, u_probe * U_ref, 'b-', label='Predicted u (m/s)')
            ax.axvspan(8, 18, alpha=0.2, color='green', label='Vents open')
            ax.set_xlabel('Time (hour of day)')
            ax.set_ylabel('u velocity (m/s)')
            ax.set_title(f'Probe (3.25, 10, 1.2) m — u at epoch {epoch}')
            ax.legend()
            ax.grid(True)
            fig.savefig(os.path.join(out_dir, f'probe_u_epoch_{epoch}.png'), dpi=150, bbox_inches='tight')
            plt.close(fig)

            # 3D view: greenhouse with flow (replacing 2D slice plots)
            nq = 15 # Increased number of points in x, y, z for denser heatmap plot grid
            xq = np.linspace(0.5, W - 0.5, nq)
            yq = np.linspace(0.5, H - 0.5, nq) # Sample y-values across the height
            zq = np.linspace(0.5, L - 0.5, nq)

            XQ, YQ, ZQ = np.meshgrid(xq, yq, zq, indexing='ij') # Use 'ij' for consistency with x,y,z iteration

            for t_sec_val, tlabel in zip(times_plot_sec, times_plot_labels):
                t_nd_val = t_sec_val / T_ref

                plot_points_x = []
                plot_points_y = []
                plot_points_z = []
                magnitude_vals = []

                # Iterate through points and filter based on roof geometry
                for i in range(XQ.shape[0]):
                    for j in range(XQ.shape[1]):
                        for k in range(XQ.shape[2]):
                            x_curr = XQ[i,j,k]
                            y_curr = YQ[i,j,k]
                            z_curr = ZQ[i,j,k]

                            # Check if the point is within the greenhouse volume (below the roof and above floor)
                            y_roof_at_x = y_roof_m(torch.tensor([x_curr], dtype=torch.float32, device=device))[0].item()

                            if y_curr < y_roof_at_x and y_curr > 0.0: # Ensure above floor (0) and below roof
                                pt = torch.tensor([[x_curr/L_ref, y_curr/L_ref, z_curr/L_ref, t_nd_val]], dtype=torch.float32, device=device)
                                with torch.no_grad():
                                    uvwp = net(pt)[0]
                                u_val, v_val, w_val = uvwp[0].item(), uvwp[1].item(), uvwp[2].item()
                                mag_val = np.sqrt(u_val**2 + v_val**2 + w_val**2) * U_ref # Convert to m/s

                                plot_points_x.append(x_curr)
                                plot_points_y.append(y_curr)
                                plot_points_z.append(z_curr)
                                magnitude_vals.append(mag_val)

                if not plot_points_x: # If no points were sampled (e.g., nq too small or geometry too restrictive)
                    print(f"No points sampled for 3D heatmap plot at {tlabel} for epoch {epoch}. Skipping plot.")
                    continue

                fig3 = plt.figure(figsize=(12, 8))
                ax3 = fig3.add_subplot(111, projection='3d')

                # Roof surface (transparent)
                x_surf_res = 30
                z_surf_res = 40
                x_surf = np.linspace(0, W, x_surf_res)
                z_surf = np.linspace(0, L, z_surf_res)
                X_surf, Z_surf = np.meshgrid(x_surf, z_surf)
                Y_surf = y_roof_m(torch.tensor(X_surf, dtype=torch.float32)).detach().cpu().numpy()
                ax3.plot_surface(X_surf, Z_surf, Y_surf, color='gray', alpha=0.4, edgecolor='black', linewidth=0.5)

                # Floor surface (transparent)
                Y_floor = np.zeros_like(X_surf)
                ax3.plot_surface(X_surf, Z_surf, Y_floor, color='gray', alpha=0.4, edgecolor='black', linewidth=0.5)

                # Walls (transparent)
                y_wall_res = 30 # Resolution for y-axis of walls
                x_wall = np.linspace(0, W, x_surf_res)
                y_wall = np.linspace(0, H, y_wall_res)
                z_wall = np.linspace(0, L, z_surf_res)

                # Wall x=0
                Y_wall_0, Z_wall_0 = np.meshgrid(y_wall, z_wall)
                X_wall_0 = np.zeros_like(Y_wall_0)
                ax3.plot_surface(X_wall_0, Z_wall_0, Y_wall_0, color='gray', alpha=0.4, edgecolor='black', linewidth=0.5)

                # Wall x=W
                Y_wall_W, Z_wall_W = np.meshgrid(y_wall, z_wall)
                X_wall_W = np.full_like(Y_wall_W, W)
                ax3.plot_surface(X_wall_W, Z_wall_W, Y_wall_W, color='gray', alpha=0.4, edgecolor='black', linewidth=0.5)

                # Wall z=0
                X_wall_0z, Y_wall_0z = np.meshgrid(x_wall, y_wall)
                Z_wall_0z = np.zeros_like(X_wall_0z)
                ax3.plot_surface(X_wall_0z, Z_wall_0z, Y_wall_0z, color='gray', alpha=0.4, edgecolor='black', linewidth=0.5)

                # Wall z=L
                X_wall_Lz, Y_wall_Lz = np.meshgrid(x_wall, y_wall)
                Z_wall_Lz = np.full_like(X_wall_Lz, L)
                ax3.plot_surface(X_wall_Lz, Z_wall_Lz, Y_wall_Lz, color='gray', alpha=0.4, edgecolor='black', linewidth=0.5)

                # Plot 3D scatter with color representing magnitude
                scatter = ax3.scatter(plot_points_x, plot_points_z, plot_points_y, # X, Z, Y coordinates
                                      c=magnitude_vals, cmap='viridis', s=20, alpha=0.7)
                plt.colorbar(scatter, ax=ax3, label='Velocity Magnitude (m/s)', shrink=0.5, aspect=10)

                ax3.set_xlabel('x (m)')
                ax3.set_ylabel('z (m)') # Swapped with y for visualization
                ax3.set_zlabel('y (m)') # Swapped with z for visualization
                ax3.set_title(f'Greenhouse Velocity Magnitude at {tlabel} — epoch {epoch}')
                ax3.set_xlim(0, W)
                ax3.set_ylim(0, L)
                ax3.set_zlim(0, H)
                fig3.savefig(os.path.join(out_dir, f'greenhouse_heatmap_3d_{tlabel.replace(":", "")}_epoch_{epoch}.png'), dpi=150, bbox_inches='tight')
                plt.close(fig3)

    print("Training done. Outputs saved to", out_dir)

if __name__ == '__main__':
    main()

Epoch 10/100  loss_phys=5.1306e-09  loss_bc=1.9034e-02  loss_ic=1.1329e-04  loss_data=0.0000e+00  loss=1.9147e-02
Epoch 20/100  loss_phys=1.2569e-08  loss_bc=1.5944e-02  loss_ic=1.7562e-04  loss_data=0.0000e+00  loss=1.6120e-02
Epoch 30/100  loss_phys=5.5660e-09  loss_bc=1.7115e-02  loss_ic=7.2996e-05  loss_data=0.0000e+00  loss=1.7188e-02
Epoch 40/100  loss_phys=1.3795e-08  loss_bc=1.6790e-02  loss_ic=1.5756e-05  loss_data=0.0000e+00  loss=1.6805e-02
Epoch 50/100  loss_phys=2.8325e-09  loss_bc=1.8078e-02  loss_ic=1.2482e-05  loss_data=0.0000e+00  loss=1.8091e-02
Epoch 60/100  loss_phys=1.2044e-09  loss_bc=1.7928e-02  loss_ic=5.3769e-06  loss_data=0.0000e+00  loss=1.7933e-02
Epoch 70/100  loss_phys=5.8021e-09  loss_bc=1.6221e-02  loss_ic=4.5250e-06  loss_data=0.0000e+00  loss=1.6226e-02
Epoch 80/100  loss_phys=3.8196e-09  loss_bc=1.7437e-02  loss_ic=3.0365e-06  loss_data=0.0000e+00  loss=1.7440e-02
Epoch 90/100  loss_phys=1.1034e-08  loss_bc=1.5876e-02  loss_ic=2.4769e-06  loss_data=0.